In [ ]:
# --- Calibration for EVI only ---
# Requirements: numpy, pandas, scikit-learn, matplotlib, scipy (for loess optional)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from sklearn.linear_model import LogisticRegression

# ==== 1) Load data ====
csv_path = r"/path/to/workspace/Complete_set/Experiments/Results/UMedPT_LR/combine_origin.csv"
df = pd.read_csv(csv_path)

# Column names (adapt if yours differ)
COL_Y_TRUE = "y_test_evi"
COL_Y_PROB = "y_pred_prob_evi"

# Keep only necessary columns and drop rows with NA
df = df[[COL_Y_TRUE, COL_Y_PROB]].dropna()

# Extract arrays
y_true = df[COL_Y_TRUE].astype(int).values
y_prob = df[COL_Y_PROB].astype(float).clip(1e-7, 1 - 1e-7).values  # clip to avoid log(0)

# ==== 2) Metrics ====
brier = brier_score_loss(y_true, y_prob)
ll = log_loss(y_true, np.c_[1 - y_prob, y_prob])   # binary
auc = roc_auc_score(y_true, y_prob)

# Calibration slope & intercept: fit y ~ logit(p)
logit_p = np.log(y_prob / (1 - y_prob)).reshape(-1, 1)
lr = LogisticRegression(solver="lbfgs", fit_intercept=True).fit(logit_p, y_true)
slope = float(lr.coef_[0, 0])
intercept = float(lr.intercept_[0])

# ECE / MCE (quantile bins recommended)
def calibration_errors(y_true, y_prob, n_bins=10):
    """
    Expected Calibration Error (ECE) and Maximum Calibration Error (MCE)
    using quantile bins (approximately equal size).
    """
    qs = np.quantile(y_prob, np.linspace(0, 1, n_bins + 1))
    qs[0], qs[-1] = 0.0, 1.0  # ensure closed range
    # digitize: assign bin index 0..n_bins-1
    bin_idx = np.digitize(y_prob, qs[1:-1], right=True)

    ece = 0.0
    mce = 0.0
    N = len(y_true)

    for b in range(n_bins):
        mask = bin_idx == b
        nb = mask.sum()
        if nb == 0:
            continue
        conf_b = y_prob[mask].mean()         # mean predicted prob
        acc_b = y_true[mask].mean()          # observed frequency
        err_b = abs(acc_b - conf_b)
        ece += (nb / N) * err_b
        mce = max(mce, err_b)
    return ece, mce

ece, mce = calibration_errors(y_true, y_prob, n_bins=10)

print(f"Brier: {brier:.4f}")
print(f"LogLoss: {ll:.4f}")
print(f"AUC: {auc:.3f}")
print(f"Calibration slope: {slope:.3f}")
print(f"Calibration intercept: {intercept:.3f}")
print(f"ECE (10 bins): {ece:.4f}")
print(f"MCE (10 bins): {mce:.4f}")

# ==== 3) Reliability / Calibration curve (quantile bins) ====
# Use scikit-learn's calibration_curve with quantile strategy (equal-frequency bins)
frac_pos, mean_pred = calibration_curve(y_true, y_prob, n_bins=10, strategy='quantile')

# ==== 4) Plot ====
plt.figure(figsize=(5, 5))
# perfect calibration
plt.plot([0, 1], [0, 1], linestyle="--", label="Perfect")
# model calibration (points + line)
plt.plot(mean_pred, frac_pos, marker="o", linewidth=1, label="Model")

plt.xlabel("Mean predicted probability (EVI)")
plt.ylabel("Observed frequency (EVI)")
plt.title("Calibration curve (EVI)")
plt.legend()
plt.tight_layout()
plt.savefig("calibration_evi.png", dpi=300)
plt.show()

# ==== (Optional) Smooth LOESS-style curve using lowess from statsmodels ====
# Uncomment if you want a smoothed curve + CI. Requires statsmodels.
# from statsmodels.nonparametric.smoothers_lowess import lowess
# sm = lowess(endog=y_true, exog=y_prob, frac=0.3, it=0, return_sorted=True)
# plt.figure(figsize=(5,5))
# plt.plot([0,1],[0,1],'--',label='Perfect')
# plt.plot(sm[:,0], sm[:,1], label='LOESS smooth')
# plt.xlabel("Predicted probability (EVI)")
# plt.ylabel("Observed frequency")
# plt.title("Calibration (LOESS) - EVI")
# plt.legend(); plt.tight_layout(); plt.savefig("calibration_evi_loess.png", dpi=300); plt.show()


In [ ]:
# --- Calibration for mfi only ---
# Requirements: numpy, pandas, scikit-learn, matplotlib, scipy (for loess optional)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from sklearn.linear_model import LogisticRegression

# ==== 1) Load data ====
csv_path = r"/path/to/workspace/Complete_set/Experiments/Results/axial_harmo_FM192/best_FM_axial_harmo.csv"
df = pd.read_csv(csv_path)

# Column names (adapt if yours differ)
COL_Y_TRUE = "ground_truth_mfi"
COL_Y_PROB = "predicted_probability_mfi"

# Keep only necessary columns and drop rows with NA
df = df[[COL_Y_TRUE, COL_Y_PROB]].dropna()

# Extract arrays
y_true = df[COL_Y_TRUE].astype(int).values
y_prob = df[COL_Y_PROB].astype(float).clip(1e-7, 1 - 1e-7).values  # clip to avoid log(0)

# ==== 2) Metrics ====
brier = brier_score_loss(y_true, y_prob)
ll = log_loss(y_true, np.c_[1 - y_prob, y_prob])   # binary
auc = roc_auc_score(y_true, y_prob)

# Calibration slope & intercept: fit y ~ logit(p)
logit_p = np.log(y_prob / (1 - y_prob)).reshape(-1, 1)
lr = LogisticRegression(solver="lbfgs", fit_intercept=True).fit(logit_p, y_true)
slope = float(lr.coef_[0, 0])
intercept = float(lr.intercept_[0])

# ECE / MCE (quantile bins recommended)
def calibration_errors(y_true, y_prob, n_bins=10):
    """
    Expected Calibration Error (ECE) and Maximum Calibration Error (MCE)
    using quantile bins (approximately equal size).
    """
    qs = np.quantile(y_prob, np.linspace(0, 1, n_bins + 1))
    qs[0], qs[-1] = 0.0, 1.0  # ensure closed range
    # digitize: assign bin index 0..n_bins-1
    bin_idx = np.digitize(y_prob, qs[1:-1], right=True)

    ece = 0.0
    mce = 0.0
    N = len(y_true)

    for b in range(n_bins):
        mask = bin_idx == b
        nb = mask.sum()
        if nb == 0:
            continue
        conf_b = y_prob[mask].mean()         # mean predicted prob
        acc_b = y_true[mask].mean()          # observed frequency
        err_b = abs(acc_b - conf_b)
        ece += (nb / N) * err_b
        mce = max(mce, err_b)
    return ece, mce

ece, mce = calibration_errors(y_true, y_prob, n_bins=10)

print(f"Brier: {brier:.4f}")
print(f"LogLoss: {ll:.4f}")
print(f"AUC: {auc:.3f}")
print(f"Calibration slope: {slope:.3f}")
print(f"Calibration intercept: {intercept:.3f}")
print(f"ECE (10 bins): {ece:.4f}")
print(f"MCE (10 bins): {mce:.4f}")

# ==== 3) Reliability / Calibration curve (quantile bins) ====
# Use scikit-learn's calibration_curve with quantile strategy (equal-frequency bins)
frac_pos, mean_pred = calibration_curve(y_true, y_prob, n_bins=10, strategy='quantile')

# ==== 4) Plot ====
plt.figure(figsize=(5, 5))
# perfect calibration
plt.plot([0, 1], [0, 1], linestyle="--", label="Perfect")
# model calibration (points + line)
plt.plot(mean_pred, frac_pos, marker="o", linewidth=1, label="Model")

plt.xlabel("Mean predicted probability (MFI)")
plt.ylabel("Observed frequency (MFI)")
plt.title("Calibration curve (MFI)")
plt.legend()
plt.tight_layout()
plt.savefig("calibration_mfi.png", dpi=300)
plt.show()

# ==== (Optional) Smooth LOESS-style curve using lowess from statsmodels ====
# Uncomment if you want a smoothed curve + CI. Requires statsmodels.
# from statsmodels.nonparametric.smoothers_lowess import lowess
# sm = lowess(endog=y_true, exog=y_prob, frac=0.3, it=0, return_sorted=True)
# plt.figure(figsize=(5,5))
# plt.plot([0,1],[0,1],'--',label='Perfect')
# plt.plot(sm[:,0], sm[:,1], label='LOESS smooth')
# plt.xlabel("Predicted probability (EVI)")
# plt.ylabel("Observed frequency")
# plt.title("Calibration (LOESS) - EVI")
# plt.legend(); plt.tight_layout(); plt.savefig("calibration_evi_loess.png", dpi=300); plt.show()
